# Results for model: meta/llama-3.1-8b-instruct

In [ ]:
import polars as pl
from xgboost import XGBRegressor
from xgboost import DMatrix
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from datetime import timedelta
import numpy as np

# Load data
df = pl.read_parquet('./jane_street_data/train.parquet')

# Define parameters
TOP_QUANTILE = 0.15
TEST_SIZE = 0.2
MAX_WINDOW_SIZE = 30

# Feature Engineering
def calculate_top_quantile(df):
    df = df.with_columns([
        pl.col('feature_00').rank().over(
            pl.window(
                pl.col('date_id'),
                range(0, MAX_WINDOW_SIZE)
            )
        ).alias('rank')
    ])
    df = df.with_columns([
        pl.when(pl.col('rank') <= (pl.col('rank').quantile(TOP_QUANTILE))).then(1).otherwise(0).alias('top_quantile')
    ])
    return df

# Create new feature
df = calculate_top_quantile(df)

# Split data into training and testing sets
train_df, test_df = train_test_split(df, test_size=TEST_SIZE, random_state=42)

# Scale data
scaler = StandardScaler()
train_df = train_df.with_columns([
    (pl.col('feature_00') - scaler.fit_transform(train_df['feature_00'])).alias('feature_00_scaled')
])
test_df = test_df.with_columns([
    (pl.col('feature_00') - scaler.transform(test_df['feature_00'])).alias('feature_00_scaled')
])

# Define XGBoost model
xgb_model = XGBRegressor(objective='reg:squarederror', max_depth=5, learning_rate=0.1, n_estimators=1000, n_jobs=-1)

# Train model
dtrain = DMatrix(train_df.select(['feature_00_scaled', 'responder_6']).to_pandas().values)
xgb_model.fit(dtrain)

# Make predictions
dtest = DMatrix(test_df.select(['feature_00_scaled', 'responder_6']).to_pandas().values)
predictions = xgb_model.predict(dtest)

# Evaluate model
mse = mean_squared_error(test_df['responder_6'], predictions)
print(f'Mean Squared Error: {mse}')